# Lab: Marketing Channel Statistical Analysis
## Part 1 — Dataset Discovery & Exploration

**Scenario:** Marketing analyst at a growing e-commerce company. The CMO wants a statistically rigorous recommendation for allocating a \$500K monthly budget across 7 channels.

---

### Dataset Documentation

| Field | Detail |
|---|---|
| **Name** | Marketing Channel Performance Dataset (Synthetic, Kaggle-style) |
| **Source** | Generated to mirror real Kaggle datasets such as *"Marketing Campaign Performance Dataset"* (kaggle.com/datasets/manishabhatt22/marketing-campaign-performance-dataset) |
| **Why chosen** | Contains 7 marketing channels with daily impressions, clicks, conversions, cost, and revenue over 90 days — exactly the multi-group, multi-metric structure the lab requires. Real Kaggle datasets in this space frequently have this layout. The synthetic version is fully reproducible without API credentials and avoids data-license concerns in a lab setting. |
| **Period** | 90 days (simulated) |
| **Channels** | Paid Search, Social Media, Email, Display, Affiliate, SEO/Organic, Influencer |

---

In [ ]:
# ── 0. Environment setup ─────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from scipy.stats import fisher_exact, false_discovery_control
import warnings
warnings.filterwarnings('ignore')

rng = np.random.default_rng(42)
sns.set_theme(style='whitegrid', palette='tab10')
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (10, 5)})

print('Libraries loaded ✓')
import scipy; print(f'SciPy {scipy.__version__}  (false_discovery_control requires ≥1.11)')

## Step 1 — Generate & Load the Dataset

We generate a realistic 90-day daily marketing dataset. Each channel has different underlying performance characteristics based on typical industry benchmarks.

In [ ]:
# ── 1. Dataset generation ────────────────────────────────────────────────────
CHANNELS = [
    'Paid Search',
    'Social Media',
    'Email',
    'Display',
    'Affiliate',
    'SEO/Organic',
    'Influencer',
]

N_DAYS = 90

# Ground-truth parameters per channel
# (impressions_mean, ctr_mean, cvr_mean, avg_order_value, daily_cost_mean)
channel_params = {
    'Paid Search':  dict(imp=18_000, ctr=0.055, cvr=0.048, aov=95,  cost=1_400),
    'Social Media': dict(imp=55_000, ctr=0.012, cvr=0.022, aov=78,  cost=900),
    'Email':        dict(imp=12_000, ctr=0.180, cvr=0.062, aov=88,  cost=120),
    'Display':      dict(imp=80_000, ctr=0.004, cvr=0.015, aov=72,  cost=600),
    'Affiliate':    dict(imp=9_000,  ctr=0.068, cvr=0.041, aov=102, cost=500),
    'SEO/Organic':  dict(imp=22_000, ctr=0.095, cvr=0.055, aov=91,  cost=180),
    'Influencer':   dict(imp=40_000, ctr=0.018, cvr=0.019, aov=115, cost=1_100),
}

rows = []
dates = pd.date_range('2024-01-01', periods=N_DAYS, freq='D')

for channel, p in channel_params.items():
    for date in dates:
        imp   = max(1, int(rng.normal(p['imp'],  p['imp']  * 0.10)))
        ctr   = np.clip(rng.normal(p['ctr'],     p['ctr']  * 0.15), 0.001, 0.60)
        clicks= max(1, int(imp * ctr))
        cvr   = np.clip(rng.normal(p['cvr'],     p['cvr']  * 0.20), 0.001, 0.50)
        conv  = max(0, int(clicks * cvr))
        aov   = max(10, rng.normal(p['aov'],     p['aov']  * 0.12))
        cost  = max(1,  rng.normal(p['cost'],    p['cost'] * 0.08))
        rev   = conv * aov
        rows.append(dict(
            date=date,
            channel=channel,
            impressions=imp,
            clicks=clicks,
            conversions=conv,
            cost=round(cost, 2),
            revenue=round(rev, 2),
        ))

df = pd.DataFrame(rows)
print(f'Shape: {df.shape}')
print(f'Channels: {df.channel.nunique()} | Days per channel: {df.groupby("channel").size().unique()[0]}')
df.head()

## Step 2 — Data Quality & Exploration

In [ ]:
# ── 2. Explore structure ─────────────────────────────────────────────────────
print('=== Data Types ===')
print(df.dtypes)
print()
print('=== Missing Values ===')
print(df.isnull().sum())
print()
print('=== Basic Statistics ===')
df.describe().round(2)

In [ ]:
# ── 3. Compute per-row derived metrics ───────────────────────────────────────
df['ctr']             = df['clicks']      / df['impressions']
df['conversion_rate'] = df['conversions'] / df['clicks'].replace(0, np.nan)
df['cpa']             = df['cost']        / df['conversions'].replace(0, np.nan)
df['roas']            = df['revenue']     / df['cost'].replace(0, np.nan)
df['profit']          = df['revenue']     - df['cost']

# Cap extreme CPA values (days with 0 conversions → inf/NaN already handled above)
df['cpa'] = df['cpa'].clip(upper=df['cpa'].quantile(0.99))

# Save cleaned dataset
df.to_csv('marketing_data.csv', index=False)
print('marketing_data.csv saved ✓')
print(f'Rows: {len(df)} | Channels: {df.channel.unique()}')
df[['channel','ctr','conversion_rate','cpa','roas','profit']].head(10).round(3)

## Step 3 — Key Marketing Metrics by Channel

In [ ]:
# ── 4. Aggregate metrics per channel ─────────────────────────────────────────
agg = df.groupby('channel').agg(
    impressions=('impressions', 'sum'),
    clicks=('clicks', 'sum'),
    conversions=('conversions', 'sum'),
    cost=('cost', 'sum'),
    revenue=('revenue', 'sum'),
).reset_index()

agg['CTR']             = (agg['clicks']      / agg['impressions'] * 100).round(2)
agg['Conversion Rate'] = (agg['conversions'] / agg['clicks']      * 100).round(2)
agg['CPA']             = (agg['cost']        / agg['conversions']).round(2)
agg['ROAS']            = (agg['revenue']     / agg['cost']).round(2)
agg['Profit']          = (agg['revenue']     - agg['cost']).round(0)
agg['Profit Margin']   = (agg['Profit']      / agg['revenue']     * 100).round(1)
agg['CPA']             = agg['CPA'].replace([np.inf, -np.inf], np.nan)
agg['ROAS']            = agg['ROAS'].replace([np.inf, -np.inf], np.nan)

print('=== Channel Performance Summary ===')
display_cols = ['channel','CTR','Conversion Rate','CPA','ROAS','Profit','Profit Margin']
print(agg[display_cols].to_string(index=False))

In [ ]:
# ── 5. Channel metrics overview bar charts ───────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
agg_sorted = agg.sort_values

metrics = [
    ('CPA',             'CPA (\$)',            False),
    ('ROAS',            'ROAS (×)',             True),
    ('Conversion Rate', 'Conversion Rate (%)',  True),
    ('conversions',     'Total Conversions',    True),
    ('cost',            'Total Cost (\$)',       False),
    ('Profit',          'Profit (\$)',           True),
]

colors = sns.color_palette('tab10', n_colors=len(CHANNELS))
channel_color = dict(zip(CHANNELS, colors))

for ax, (metric, label, higher_better) in zip(axes.flat, metrics):
    data = agg.dropna(subset=[metric]).sort_values(metric, ascending=higher_better)
    bar_colors = [channel_color[c] for c in data['channel']]
    bars = ax.barh(data['channel'], data[metric], color=bar_colors, edgecolor='white')
    if metric == 'Profit':
        ax.axvline(0, color='black', linewidth=1.2, linestyle='--')
    ax.set_xlabel(label)
    ax.set_title(label)
    ax.tick_params(axis='y', labelsize=9)

plt.suptitle('Marketing Channel Performance Overview', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('group_metrics_overview.png', bbox_inches='tight')
plt.show()
print('group_metrics_overview.png saved ✓')

In [ ]:
# ── 6. Distribution plots (daily metrics per channel) ────────────────────────
daily = df.dropna(subset=['cpa','conversion_rate','roas']).copy()
daily = daily[daily['cpa'] < daily['cpa'].quantile(0.99)]  # drop extreme outliers

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (col, label) in zip(axes, [
    ('cpa',             'Daily CPA (\$)'),
    ('conversion_rate', 'Daily Conversion Rate'),
    ('roas',            'Daily ROAS'),
]):
    for channel in CHANNELS:
        vals = daily.loc[daily['channel'] == channel, col].dropna()
        ax.hist(vals, bins=20, alpha=0.45, label=channel, density=True)
    ax.set_xlabel(label)
    ax.set_ylabel('Density')
    ax.set_title(f'Distribution of {label}')
    ax.legend(fontsize=7)

plt.suptitle('Daily Metric Distributions by Channel', fontsize=13)
plt.tight_layout()
plt.savefig('group_distributions.png', bbox_inches='tight')
plt.show()
print('group_distributions.png saved ✓')

In [ ]:
# ── 7. Box plots ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, (col, label) in zip(axes, [
    ('cpa',  'Daily CPA (\$)'),
    ('roas', 'Daily ROAS'),
]):
    data_plot = daily[['channel', col]].dropna()
    sns.boxplot(data=data_plot, x='channel', y=col, ax=ax,
                palette='tab10', flierprops=dict(marker='.', alpha=0.4))
    ax.set_xlabel('')
    ax.set_ylabel(label)
    ax.set_title(f'{label} by Channel')
    ax.tick_params(axis='x', rotation=25)

plt.tight_layout()
plt.savefig('group_distributions.png', bbox_inches='tight')  # overwrite with box plots appended
plt.show()
print('Box plots added to group_distributions.png ✓')

## Key Insights from Exploration

| Channel | CPA | ROAS | Conv. Rate | Note |
|---|---|---|---|---|
| Email | Lowest | Highest | Highest | Cheapest to convert, very efficient |
| SEO/Organic | Low | High | High | Good efficiency, zero media cost |
| Paid Search | Medium | Medium-High | Medium-High | Strong intent signal |
| Affiliate | Medium | Medium | Medium | Worth monitoring |
| Social Media | Higher | Lower | Lower | High volume, low precision |
| Influencer | High | Low | Lowest | High CPM, poor conversion |
| Display | Highest CPA | Lowest ROAS | Lowest | Awareness play, not conversion |

**Statistical rigour is required** before acting on these visual patterns — Part 2 will test whether differences are real or noise.